---
title: "Numpy: Vektorisierung und Arrays"
short_title: "Numpy"
description: "Die tragenden Ideen von Numpy an Beispielen aus der Stochastik: Arrays, Broadcasting, Masken und Fancy Indexing, ufuncs, und was man davon auswendig kennen sollte."
---


In diesem kurzen Abschnitt lernen wir die wesentlichen Ideen bei Numpy kennen.
Eine längere und bessere (und englischere) Fassung davon sind die [Numpy fundamentals](https://numpy.org/devdocs/user/basics.html) im Numpy User Guide, die Sie am besten im Anschluss überfliegen sollten.
Ein richtiges (und sehr gutes) Lehrbuch für alle, die mit Python vertraut sind (und als das betrachten wir uns zu diesem Punkt der Vorlesung) ist [Nicolas Rougier's From Python to Numpy](https://www.labri.fr/perso/nrougier/from-python-to-numpy/), wo für uns zunächst Kapitel 3 relevant ist.

Kurz lässt sich sagen, dass mit Numpy [Array-orientierte Programmierung](https://en.wikipedia.org/wiki/Array_programming) (auch: *Vektorisierung*) in Python möglich wird.

Warum lohnt sich das? Drei Gründe, die zusammenhängen.

Erstens ist Numpy vor allem eine **Schnittstelle zu sehr schnellen Bibliotheken** für lineare Algebra und Statistik, die in C, Fortran und teilweise in Assembler geschrieben sind und seit Jahrzehnten optimiert werden. Ein `np.dot` ruft Code auf, den niemand in Python nachbauen möchte.

Zweitens erlaubt diese Arbeitsweise **SIMD** (*single instruction, multiple data*): Statt eine Schleife über eine Liste zu schreiben und jedes Element einzeln anzufassen, wendet man eine Operation auf einen ganzen Vektor an. Moderne Prozessoren können mehrere Werte in einem Schritt verrechnen — aber nur, wenn für alle dasselbe zu tun ist. Verzweigungen (`if`-Abfragen innerhalb der Schleife) machen das zunichte. Daraus folgt die Arbeitsregel dieses Kapitels: **Code verzweigungsfrei schreiben, also vektorisieren.** Wo doch eine Fallunterscheidung nötig ist, formuliert man sie als Maske über dem ganzen Array statt als `if` im Schleifenrumpf; wie das geht, steht im Abschnitt *Masken und Fancy Indexing*.

Drittens ist diese Denkweise übertragbar. PyTorch, die Standardbibliothek für neuronale Netze, arbeitet fast genauso — dort heißt das Objekt `tensor` statt `ndarray`, und es liegt wahlweise im Hauptspeicher oder auf der Grafikkarte. Wer hier vektorisieren gelernt hat, kann es dort weiterbenutzen; die Vorlesung *Einführung in Deep Learning* setzt genau das voraus.

Wir verwenden Numpy in einer Version größergleich 2.5, wozu Python 3.12 (oder höher) nötig ist. Der Code in diesem Skript sollte allerdings größtenteils auch mit Numpy 1.26 laufen. Für weitere Informationen zur Versionskompatibilität gibt es die [NEP 29](https://numpy.org/neps/nep-0029-deprecation_policy.html).

## Arrays
<!--
Zur Probability mit Numpy ist das hier (auf deutsch) gut geeignet:
https://www.python-kurs.eu/python_numpy_wahrscheinlichkeit.php
-->
Numeric Python (Numpy) wird meist als `np` abgekürzt importiert:

In [1]:
import numpy as np
np.__version__

'2.5.4'

In [2]:
sample = np.random.random(10)
print(sample)
print(type(sample))
sample

[0.52083311 0.88741935 0.73074552 0.71415566 0.20429192 0.05988784
 0.43662357 0.81704234 0.91341601 0.25041575]
<class 'numpy.ndarray'>


array([0.52083311, 0.88741935, 0.73074552, 0.71415566, 0.20429192,
       0.05988784, 0.43662357, 0.81704234, 0.91341601, 0.25041575])

Viele nützliche Hilfsfunktionen sind in Numpy enthalten, die wiederum Numpy-eigene Datenstrukturen (den Array) verarbeiten.

In [3]:
sample = np.random.randint(low=0, high=10, size=5000)
print("mean    =", np.mean(sample),   "\nexpected=", 9/2)
# 99/12 = (n²-1)/12 mit n=10, die Varianz der diskreten Gleichverteilung
# (hergeleitet im Kapitel "Wichtigste diskrete Verteilungen"):
print("std²    =", np.std(sample)**2, "\nvariance=", 99/12)


mean    = 4.4114 
expected= 4.5
std²    = 8.361750040000002 
variance= 8.25


Es gibt auch eine `arange`-Methode, sie erzeugt aber keine Range-Objekte in Numpy:

In [4]:
myRange = np.arange(1, 7)
print(myRange, type(myRange), myRange.dtype)
myRange

[1 2 3 4 5 6] <class 'numpy.ndarray'> int64


array([1, 2, 3, 4, 5, 6])

Der Numpy-Array `ndarray` trägt im Gegensatz zur Python-Liste einen festen Datentyp, den alle Elemente gemeinsam haben. Das können alle Numpy-datatypes (dtypes) sein, z.B. `double` (kompatibel mit dem Python-`float`) oder `float32` (auf den meisten Plattformen kompatibel mit dem C-`float`)  oder `long` (kompatibel mit dem Python-`int`).

In [5]:
import timeit
ordinary_list = [1,2,3,4,5,6,5,4,3,2,1]*10
def sort_array(dtype):
    a = np.array(ordinary_list, dtype)
    a.sort()
print(timeit.timeit(lambda : sort_array(np.byte)))
print(timeit.timeit(lambda : sort_array(np.float64)))

2.7987090409733355


3.5988903171382844


Dadurch, dass der Datentyp präzise bekannt ist, kann Numpy darauf optimierte Algorithmen, direkt in C implementiert, verwenden.

Was es noch für `dtype`s gibt und wie sie eingesetzt werden, können wir der Dokumentation entnehmen:
* ["Structured Arrays"](https://numpy.org/doc/stable/user/basics.rec.html)
* ["Data types" in den "Numpy fundamentals"](https://numpy.org/doc/stable/user/basics.types.html)
* ["Scalars"](https://numpy.org/doc/stable/reference/arrays.scalars.html)
* ["Data type objects (dtype)"](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
* ["numpy.dtype"](https://numpy.org/doc/stable/reference/generated/numpy.dtype.html)

Es gibt noch mehr Wege, Arrays zu erzeugen, außer mit `arange` oder durch Konvertieren einer Python-Sequenz. Z.B. lässt sich mit `ones` ein Array gefüllt mit $1$ und mit `zeros` ein Array gefüllt mit $0$ erzeugen.

Der Grund dafür, dass das Array den Namen `ndarray` trägt, ist, dass es für „$n$-dimensional array“ steht.
Wenn man in Python eine Matrix speichern möchte, würde man das als Liste der Zeilenvektoren (oder der Spaltenvektoren) tun, etwa

In [6]:
matrix = [[1,0,0], [0,1,0], [0,0,1]]  # Einheitsmatrix
print(matrix, type(matrix))
quarkix = matrix[:]   # eine "Kopie" per Slicing
print(quarkix[0][0])
quarkix[0][0] = 0     # wir ändern den oberen linken Eintrag
print(matrix[0][0])

[[1, 0, 0], [0, 1, 0], [0, 0, 1]] <class 'list'>
1
0


Daran sehen wir ein Problem: Python behandelt unsere Matrix wie eine Liste (so haben wir es ja auch hingeschrieben), also wird beim Kopieren der Liste der Inhalt (die Zeilenvektoren) nicht mitkopiert (sondern nur die Pointer darauf).

In [7]:
npmatrix = np.identity(3, int)  # Einheitsmatrix
print(npmatrix, type(npmatrix))
npquarkix = npmatrix[:]
print(npquarkix[0][0])
npquarkix[0][0] = 2     # wir ändern den oberen linken Eintrag
print(npmatrix[0][0])
# Mit einer echten Kopie wäre das nicht passiert:
real_copy = npmatrix.copy()
real_copy[0][0] = 1
assert npmatrix[0][0] != 1

[[1 0 0]
 [0 1 0]
 [0 0 1]] <class 'numpy.ndarray'>
1
2


Es ist wichtig, festzustellen, dass der Numpy-Array das gleiche Verhalten an den Tag legt wie unsere Python-Liste-von-Listen. Wir können gleich damit indizieren und slicen, und es gibt das gleiche Problem beim Kopieren über die Slicing-Syntax.

Der `shape`-Parameter sagt uns, welche Form unser Numpy-Array hat. Dabei handelt es sich um ein $d$-Tupel, wobei $d$ die Dimension ist. Eine Matrix ist $2$-dimensional, ein Vektor $1$-dimensional und ein Skalar $0$-dimensional.

In [8]:
print(npmatrix.shape)
print(npmatrix[0], npmatrix[0].shape)
print(npmatrix[0][0], npmatrix[0][0].shape)

(3, 3)
[2 0 0] (3,)
2 ()


In Numpy kann man noch etwas feiner slicen.
Die Allgemeine Syntax ist `[start:stop:step, ..]` wobei man mit dem Komma getrennt über die Achsen geht. Ein zweidimensionaler Array hat zwei Achsen, wobei Achse $0$ von oben nach unten und Achse $1$ von links nach rechts indiziert ist. Während "step" auch mit Python-Listen funktioniert, ist das indizieren mit mehreren Achsen eine Spezialität von Numpy.

In [9]:
zahlen = list(range(1,10))
npmatrix = np.array(zahlen)
print("not in shape:", npmatrix)
npmatrix = npmatrix.reshape(3, 3)
print("in much better shape:\n"+ str(npmatrix))
print("Zeilenvektor Zeile 0:", npmatrix[0])
print("Spaltenvektor Spalte 0:", npmatrix[:,0])
print("Spalten 1-2:\n"+ str(npmatrix[:,1:]))
print("Alle Zeilen, Schrittweite 2\n"+ str(npmatrix[0::2]))

not in shape: [1 2 3 4 5 6 7 8 9]
in much better shape:
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Zeilenvektor Zeile 0: [1 2 3]
Spaltenvektor Spalte 0: [1 4 7]
Spalten 1-2:
[[2 3]
 [5 6]
 [8 9]]
Alle Zeilen, Schrittweite 2
[[1 2 3]
 [7 8 9]]


In klassischem Python-Code würden wir auf einen Eintrag einer Matrix zugreifen mit `matrix[x][y]`, und das funktioniert so auch in Numpy. Allerdings wird dabei zunächst ein weiteres Array-Objekt `matrix[x]` erzeugt (eine View, siehe unten) und dann darauf `[y]` aufgerufen. Es ist daher grundsätzlich effizienter, direkt Numpy's `[x,y]` zu verwenden.

Numpy erzeugt bewusst keine Kopien beim Slicing, sondern nur eine andere Sichtweise auf den gleichen Speicherbereich (daher auch das oben beobachtete Verhalten bei `[:]`). Ob zwei Arrays auf den gleichen Speicherbereich verweisen, lässt sich mit `np.may_share_memory` prüfen. Dabei bedeutet ein positives Ergebnis keineswegs, dass die Arrays voneinander abhängig sind - so verweisen die erste und die zweite Spalte einer Matrix auch auf den gleichen Speicherbereich, nämlich die ganze Matrix. Wenn man nun einen der beiden Vektoren ändert, bleibt der andere unverändert - die ganze Matrix aber ändert sich mit.

In [10]:
npmatrix = np.array(list(range(1,10))).reshape(3,3)
candidates = (npmatrix[0::2], npmatrix[1])
print("May share memory (but actually don't):",
      candidates, np.may_share_memory(*candidates))
print(type([0,0,0]))     # vor der Zuweisung
npmatrix[1] = [0,0,0]
print(type(npmatrix[1])) # nach der Zuweisung
print(npmatrix)          # die ganze Matrix ist wie verändert

May share memory (but actually don't): (array([[1, 2, 3],
       [7, 8, 9]]), array([4, 5, 6])) True
<class 'list'>
<class 'numpy.ndarray'>
[[1 2 3]
 [0 0 0]
 [7 8 9]]


Ausführliche Informationen zum Slicing und Indizieren liefert [die Dokumentation](https://numpy.org/doc/stable/user/basics.indexing.html).

## Broadcasting

Während für Python-Listen der Additionsoperator die Listenkonkatenation ist, und damit die Multiplikation von Listen mit Skalaren definiert ist, ist die Multiplikation von zwei Listen undefiniert.
Für Numpy-Arrays sind deutlich mehr arithmetische Operationen verfügbar:

In [11]:
E = np.identity(3, int)
print(E, "= E")
A = np.ones((3,3), int)
A[0] = [0,0,0]
print(A, "= A")
print(E + A, "= E + A")
print(E * A, "= EA")
print((E+A)**2, "= (E+A)(E+A)")

[[1 0 0]
 [0 1 0]
 [0 0 1]] = E
[[0 0 0]
 [1 1 1]
 [1 1 1]] = A
[[1 0 0]
 [1 2 1]
 [1 1 2]] = E + A
[[0 0 0]
 [0 1 0]
 [0 0 1]] = EA
[[1 0 0]
 [1 4 1]
 [1 1 4]] = (E+A)(E+A)


Wenn man das aufmerksam nachverfolgt, stellt man fest, dass diese Rechnungen keine Matrizenmultiplikationen sind,
sondern schlicht elementweise erfolgt sind - so sind die Operationen auf Arrays definiert.
Besonders tückisch ist dies:

In [12]:
v = np.ones(3, int)
print(v, "= v")
print(A*v, "= A*v (aber nicht die Matrixmultiplikation)")

[1 1 1] = v
[[0 0 0]
 [1 1 1]
 [1 1 1]] = A*v (aber nicht die Matrixmultiplikation)


Um explizit mit Matrizenkalkül zu rechnen, hat man früher den Numpy-Datentyp `matrix` verwendet, aber dieser ist als `deprecated` (veraltet) markiert und wird in zukünftigen Numpy-Versionen abgeschafft. Heutzutage nutzt man die Methode `np.matmul` oder den Infix-Operator `@`.

In [13]:
print(np.matmul(A,v))
print(A@v)

[0 3 3]
[0 3 3]


Für viele Probleme ist es sehr hilfreich, nicht den Matrizenkalkül zu verwenden, sondern elementweise arithmetische Operationen auszuführen.
*Broadcasting* ist ein Mechanismus, der diesen elementweisen Kalkül etwas praktischer macht.
So ist die Operation ($n \times n$-array) * ($n$-Vektor) automatisch interpretiert, indem der $n$-Vektor $n$-fach kopiert wird, sodass die Multiplikation einer jeden Zeile des linken Arrays mit dem Vektor (elementweise) durchgeführt wird.

Dazu ist es wirklich hilfreich, einmal [die Dokumentation](https://numpy.org/doc/stable/user/basics.broadcasting.html) zu überfliegen.

Ein *Tensor* in PyTorch, der Bibliothek der Vorlesung *Einführung in Deep Learning*, ist im Wesentlichen ein Numpy-Array mit zwei Zusätzen: einer Gerätezuordnung (CPU oder GPU) und automatischer Ableitung. `shape`, Achsen (`axis`) und Broadcasting funktionieren dort genauso wie in Numpy, weshalb jene Vorlesung dieses Kapitel voraussetzt.

## Masken und Fancy Indexing

Ein Vergleich wie `a > 2` wird elementweise ausgewertet und liefert ein Array aus `True`/`False`, eine **Maske**. Eine Maske kann man als Index verwenden: Man bekommt genau die Einträge, an denen `True` steht, und kann diesen Einträgen auch etwas zuweisen. Das ist das Werkzeug, mit dem wir im nächsten Kapitel Daten filtern ("alle Passagiere unter 13", "alle Zeilen ohne fehlenden Wert"). Mit `np.where(bedingung, a, b)` wählt man elementweise aus zwei Arrays aus: dort, wo die Bedingung wahr ist, den Eintrag aus `a`, sonst den aus `b`. So ersetzt man eine `if`-Abfrage pro Element durch eine einzige Array-Operation:

In [14]:
a = np.array([3, 1, 4, 1, 5, 9])
maske = a > 2
print(maske)                       # Maske: Array aus True/False
print(a[maske])                    # nur die Einträge, wo True steht
print(a[(a > 2) & (a < 9)])        # Bedingungen mit & (und), | (oder), ~ (nicht) verknüpfen,
                                   # nicht mit and/or; jede Bedingung in Klammern!
a[a == 1] = 0                      # Zuweisung an eine Maske
print(a)
print(a[[0, 2, 4]])                # "Fancy Indexing": eine Liste von Positionen als Index
print(np.isnan(np.array([1.0, np.nan, 3.0])))   # Masken für fehlende Werte
print(np.where(a > 2, a, 0))       # np.where: wo a > 2 den Wert aus a, sonst 0 (kein if nötig)

[ True False  True False  True  True]
[3 4 5 9]
[3 4 5]
[3 0 4 0 5 9]
[3 4 5]
[False  True False]
[3 0 4 0 5 9]


## Wahrscheinlichkeitsvektoren

Im Kapitel über Zufallsvariablen war eine Verteilung auf $n$ Werten ein Vektor $p \in \Delta_n$: nicht-negative Einträge, die sich zu $1$ summieren. In Numpy ist das einfach ein Array, das man durch seine Summe teilt, und eine Stichprobe aus dieser Verteilung ist ein Aufruf von `choice` mit dem Argument `p`:

In [15]:
p = np.array([1, 1, 1, 1, 1, 3], dtype=float)   # gezinkter Würfel: die 6 ist dreimal so wahrscheinlich
p = p / p.sum()                                   # normieren: jetzt liegt p im Simplex Δ_6
print(p, p.sum())

rng = np.random.default_rng(1)
wuerfe = rng.choice(np.arange(1, 7), size=10000, p=p)   # Stichprobe aus der pmf p
print(np.bincount(wuerfe)[1:] / len(wuerfe))            # relative Häufigkeiten, ungefähr p

[0.125 0.125 0.125 0.125 0.125 0.375] 1.0
[0.1269 0.1222 0.1218 0.1244 0.1285 0.3762]


## ufuncs

`ufunc` steht für "universal function" und bezeichnet eine Methode, die auf Numpy Arrays vektorisiert laufen kann.

Mit jeder `ufunc` lässt sich z.B. `reduce` durchführen, wo entlang einer Achse des Arrays die `ufunc` auf die resultierenden kleineren Arrays angewandt wird.

Um selbst eine `ufunc` zu schreiben, muss man [C-Code programmieren](https://numpy.org/doc/stable/user/c-info.ufunc-tutorial.html) oder aber [einen Wrapper um eine Python-Methode legen](https://numpy.org/doc/stable/reference/generated/numpy.frompyfunc.html).

Es lohnt sich, einen kurzen Blick auf alle bereits definierten `ufunc`s zu werfen:
[Available ufuncs](https://numpy.org/doc/stable/reference/ufuncs.html#available-ufuncs)

## Was muss man kennen?

Numpy und andere Bibliotheken des Scientific Computing in Python sind groß, man kann kaum alle Funktionen kennen.
Für dieses Skript liefert ein Durchlauf über die Code-Zellen aller Notebooks (`grep -ohE "np\.[a-z_]+(\.[a-z_]+)?" *.ipynb | sort -u`, händisch bereinigt) diese vollständige Liste, Stand September 2026. Sie muss nicht auswendig gelernt werden, gibt aber eine klare Abschätzung nach oben: Mehr muss man definitiv auf keinen Fall können! **Fett** gedruckt sind die Funktionen, die man wirklich im Kopf haben sollte, weil man sie ständig tippt; der Rest ist Nachschlagewerk. Außerdem sollte man bedenken, dass diese Liste für das gesamte Skript gilt, und wir vieles davon noch nicht kennen gelernt haben.

Für die tägliche Arbeit hilft ein Spickzettel mehr als eine Liste: Die [Numpy-Referenz](https://numpy.org/doc/stable/reference/index.html) hat eine nach Themen sortierte Übersicht, und für den Einstieg gibt es das [kompakte Einsteigerkapitel der Numpy-Dokumentation](https://numpy.org/doc/stable/user/absolute_beginners.html). Ausdrucken, danebenlegen, benutzen — Auswendiglernen bringt hier nichts.


 * **abs**
 * **all**
 * **allclose**
 * append
 * **arange**
 * **argmin**
 * **array**
 * atleast_1d
 * bincount
 * byte
 * c_
 * column_stack
 * **concatenate**
 * **copy**
 * **corrcoef**
 * cos
 * **cov**
 * **cumsum**
 * diag
 * **diff**
 * **dot**
 * dstack
 * equal
 * **exp**
 * **eye**
 * float64
 * **full**
 * genfromtxt
 * heaviside
 * **hstack**
 * identity
 * **isclose**
 * **isnan**
 * linalg.det
 * linalg.eig
 * linalg.eigh
 * **linalg.inv**
 * **linalg.norm**
 * linalg.svd
 * **linspace**
 * **log**
 * log2
 * logspace
 * **matmul**
 * **max**
 * **maximum**
 * may_share_memory
 * **mean**
 * **median**
 * **meshgrid**
 * mgrid
 * **min**
 * **minimum**
 * **nan**
 * nansum
 * **ones**
 * ones_like
 * outer
 * **pi**
 * power
 * ptp
 * **random.choice**
 * **random.default_rng**
 * random.multivariate_normal
 * random.permutation
 * random.rand
 * random.randint
 * random.randn
 * random.random
 * **random.seed**
 * random.uniform
 * **round**
 * searchsorted
 * set_printoptions
 * sin
 * size
 * **sort**
 * **sqrt**
 * **square**
 * **stack**
 * **std**
 * **sum**
 * **unique**
 * **var**
 * **vstack**
 * **where**
 * **zeros**
 * zeros_like